# From eclipse events to labeled days

If we train directly on the catalog, every example has the same answer (i.e everyday is an eclipse). We need days on which an eclipse occurs and days on which the Sun and Moon pass one another in the sky without producing an eclipse. There are two different tables to build. The first assigns a label to every calendar day. The second selects the much smaller set of observing windows for which we obtain physical measurements.

Keep the four CSV files from Data Extraction in the working directory. Completing the vector table requires Horizons access.

In [ ]:
%pip install numpy pandas scipy requests

In [ ]:
import numpy as np
import pandas as pd
import scipy
import requests

In [ ]:
df_A=pd.read_csv("dataset_A.csv")
df_A.head()

## On which day did greatest eclipse occur?

Let $t_{\rm TD}$ be the catalog clock time in seconds since midnight. The corresponding UT reading is $t_{\rm TD}-\Delta T$. If it falls outside $[0,86400)$, the integer day must change as well. A Julian day number gives us an integer on which we can perform this shift across a month or year boundary. Here the catalog convention changes from Julian to Gregorian in October 1582. The output labels use proleptic Gregorian dates.

In [ ]:
df_A["eclipse"]=1
# 1. Convert month names to numbers
months = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

df_A["month_num"] = (
    df_A["month"].str.strip().str.title().map(months)
)

if df_A["month_num"].isna().any():
    raise ValueError("Some month values could not be converted.")


# 2. Convert NASA dates to Julian Day Numbers
def to_jdn(year, month, day):
    a = (14 - month) // 12
    y = year + 4800 - a
    m = month + 12 * a - 3

    if (year, month, day) >= (1582, 10, 15):
        return (day + (153*m + 2)//5 + 365*y
                + y//4 - y//100 + y//400 - 32045)

    return day + (153*m + 2)//5 + 365*y + y//4 - 32083


# 3. Convert Julian Day Numbers to Gregorian dates
def to_gregorian(jdn):
    a = jdn + 32044
    b = (4*a + 3) // 146097
    c = a - (146097*b)//4
    d = (4*c + 3) // 1461
    e = c - (1461*d)//4
    m = (5*e + 2)//153

    day = e - (153*m + 2)//5 + 1
    month = m + 3 - 12*(m//10)
    year = 100*b + d - 4800 + m//10

    return year, month, day


df_A["jdn"] = [
    to_jdn(int(y), int(m), int(d))
    for y, m, d in df_A[["year", "month_num", "day"]].itertuples(
        index=False, name=None
    )
]


# 4. Convert TD to UT and correct the date
td_seconds = pd.to_timedelta(df_A["td_ge"]).dt.total_seconds()

day_shift = np.floor(
    (td_seconds - pd.to_numeric(df_A["dt"])) / 86400
).astype(int)

dates = [
    to_gregorian(int(jdn + shift))
    for jdn, shift in zip(df_A["jdn"], day_shift)
]

df_A["date"] = pd.PeriodIndex.from_fields(
    year=[y for y, m, d in dates],
    month=[m for y, m, d in dates],
    day=[d for y, m, d in dates],
    freq="D"
)

Having assigned each event to a UT day, we can fill the interval between the first and last event. A one identifies the day of greatest eclipse, even for a partial event. Other days receive zero. This convention matters when a path crosses midnight.

In [ ]:
# 5. Generate all dates within the dataset interval
all_dates = pd.period_range(
    df_A["date"].min(),
    df_A["date"].max(),
    freq="D"
)


# 6. Generate daily dataset and fill missing eclipse dates with zero
df_A_daily = (
    df_A.groupby("date")["eclipse"]
    .max()
    .reindex(all_dates, fill_value=0)
    .rename_axis("date")
    .reset_index()
)


# 7. Check the resulting dataset
print("Total days:", len(df_A_daily))
print("Days with eclipses:", (df_A_daily["eclipse"] == 1).sum())
print("Days without eclipses:", (df_A_daily["eclipse"] == 0).sum())
df_A_daily.head()
df_A_daily.to_csv("dataset_A_daily.csv", index=False)

In [ ]:
df_A_daily.head()

## Which of those days need vectors?

Restrict the sampling to the span of the path tables. For each central eclipse we keep the C timestamps. For a partial eclipse we have no central line to follow, so we place a two-minute grid around greatest eclipse and use the median duration of the available path windows. The previous and following days give us examples close to alignment. Three consecutive days drawn between events also sample the intervening orbit.

Within each group we shift the observing window by whole days. Thus the label belongs to `sample_day`, whereas `datetime_ut` identifies the instant at which to query JPL. They can differ near midnight. The fixed seed makes the control windows reproducible.

In [ ]:
df_A=pd.read_csv("dataset_A.csv")
df_C=pd.read_csv("dataset_C.csv")
df_D=pd.read_csv("dataset_D.csv")

STATE_COLS = [
    f"{variable}_{body}"
    for body in ("sun", "moon")
    for variable in ("x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms")
]


def build_df_jpl(df_A, df_C, df_D, seed=42):

    months = {
        m: i for i, m in enumerate(
            ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1
        )
    }

    a = df_A.copy()
    c = df_C.copy()
    d = df_D.copy()

    required_a = {"year", "month", "day", "td_ge", "dt", "eclipse_type"}
    required_c = {"event_id", "time_ut"}
    required_d = {"event_id", "datetime_ut", *STATE_COLS}

    if (
        not required_a.issubset(a.columns)
        or not required_c.issubset(c.columns)
        or not required_d.issubset(d.columns)
    ):
        raise ValueError("Missing required columns in df_A, df_C or df_D")

    # 1. Prepare Dataset A
    a = a.loc[a["year"].between(1990, 2051)].copy()

    a["month_int"] = (
        a["month"].astype(str).str.strip().str.title().map(months)
    )

    if a["month_int"].isna().any():
        raise ValueError("Invalid month values")

    a["date_td"] = pd.to_datetime(
        dict(
            year=a["year"],
            month=a["month_int"],
            day=a["day"]
        )
    )

    a["greatest_ut"] = (
        a["date_td"]
        + pd.to_timedelta(a["td_ge"])
        - pd.to_timedelta(pd.to_numeric(a["dt"]), unit="s")
    )

    a["eclipse_day"] = a["greatest_ut"].dt.normalize()

    all_eclipse_days = set(a["eclipse_day"])

    # 2. Validate datasets C and D
    d["datetime_ut"] = pd.to_datetime(d["datetime_ut"])

    if d.duplicated(["event_id", "datetime_ut"]).any():
        raise ValueError("Duplicated observations in D")

    if not c[["event_id", "time_ut"]].equals(
        d[["event_id", "time_ut"]]
    ):
        raise ValueError("Datasets C and D do not match")

    c_ids = c["event_id"].drop_duplicates()

    if not c_ids.str.fullmatch(
        r"SE\d{4}[A-Za-z]{3}\d{2}[ATH]"
    ).all():
        raise ValueError("Invalid event_id format")

    c_event_dates = pd.to_datetime(
        c_ids.str.slice(2, 11),
        format="%Y%b%d"
    )

    if c_event_dates.duplicated().any():
        raise ValueError("Duplicated eclipse dates in C")

    id_by_date = dict(zip(c_event_dates, c_ids))

    first_day = min(c_event_dates)
    last_day = max(c_event_dates)

    # 3. Select all eclipses within the C interval
    a = a[
        a["eclipse_day"].between(first_day, last_day)
    ].sort_values("greatest_ut").copy()

    if a["eclipse_day"].duplicated().any():
        raise ValueError("More than one eclipse on the same day")

    a["has_c_trajectory"] = a["date_td"].isin(id_by_date)

    non_c = a[~a["has_c_trajectory"]]

    if not non_c["eclipse_type"].astype(str).str.startswith("P").all():
        raise ValueError("Non-partial eclipses missing from C")

    a["event_id"] = [
        id_by_date[date]
        if date in id_by_date
        else f"SE{date.strftime('%Y%b%d')}P"
        for date in a["date_td"]
    ]

    # 4. Preserve original sampling times from D
    original_times = {
        eid: group["datetime_ut"].reset_index(drop=True)
        for eid, group in d.groupby("event_id", sort=False)
    }

    if set(original_times) != set(c_ids):
        raise ValueError("Event IDs from C and D do not match")

    lengths = d.groupby("event_id")["datetime_ut"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 60
    )

    median_span_minutes = int(
        np.round(lengths.median() / 2) * 2
    )

    # 5. Define sampling windows for partial eclipses
    partial_offsets = pd.to_timedelta(
        np.arange(median_span_minutes // 2 + 1) * 2
        - median_span_minutes / 2,
        unit="m"
    )

    rng = np.random.default_rng(seed)

    parts = []
    previous_day = None

    # 6. Generate positive and negative sampling windows
    for row in a.itertuples(index=False):

        eclipse_day = row.eclipse_day
        eid = row.event_id

        if row.has_c_trajectory:
            positive_times = original_times[eid]
        else:
            positive_times = pd.Series(
                row.greatest_ut + partial_offsets
            )

        def add_window(day, kind, label):

            shifted_times = (
                positive_times + (day - eclipse_day)
            )

            parts.append(
                pd.DataFrame({
                    "event_id": eid,
                    "reference_day": eclipse_day,
                    "sample_day": day,
                    "sample_type": kind,
                    "eclipse": np.int8(label),
                    "eclipse_type": row.eclipse_type,
                    "has_c_trajectory": (
                        bool(row.has_c_trajectory) and label == 1
                    ),
                    "datetime_ut": shifted_times
                })
            )

        # Positive eclipse window
        add_window(eclipse_day, "eclipse", 1)

        # Previous and next day
        for delta, kind in [
            (-1, "previous_day"),
            (1, "next_day")
        ]:

            day = eclipse_day + pd.Timedelta(days=delta)

            if day in all_eclipse_days:
                raise ValueError(
                    f"Negative control overlaps an eclipse: {day.date()}"
                )

            add_window(day, kind, 0)

        # Three consecutive random days between eclipses
        if previous_day is not None:

            candidates = pd.date_range(
                previous_day + pd.Timedelta(days=2),
                eclipse_day - pd.Timedelta(days=4),
                freq="D"
            )

            valid_starts = [
                s for s in candidates
                if all(
                    (s + pd.Timedelta(days=k))
                    not in all_eclipse_days
                    for k in range(3)
                )
            ]

            if not valid_starts:
                raise ValueError(
                    f"No available three-day interval between "
                    f"{previous_day} and {eclipse_day}"
                )

            first_random = valid_starts[
                int(rng.integers(len(valid_starts)))
            ]

            for k in range(3):
                add_window(
                    first_random + pd.Timedelta(days=k),
                    f"random_day_{k+1}",
                    0
                )

        previous_day = eclipse_day

    # 7. Combine all sampling windows
    df_jpl = pd.concat(parts, ignore_index=True)

    if df_jpl.duplicated(["datetime_ut"]).any():
        raise ValueError("Duplicated timestamps in df_jpl")

    # 8. Reuse available vectors from Dataset D
    existing = d[
        ["event_id", "datetime_ut", *STATE_COLS]
    ]

    df_jpl = df_jpl.merge(
        existing,
        on=["event_id", "datetime_ut"],
        how="left",
        validate="m:1"
    )

    # 9. Identify observations requiring JPL Horizons
    df_jpl["vectors_available"] = (
        df_jpl[STATE_COLS].notna().all(axis=1)
    )

    df_jpl["needs_jpl"] = ~df_jpl["vectors_available"]

    # 10. Sort and format the dataset
    df_jpl = df_jpl.sort_values(
        ["datetime_ut", "event_id"]
    ).reset_index(drop=True)

    df_jpl["datetime_ut"] = (
        df_jpl["datetime_ut"].dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    df_jpl["reference_day"] = (
        df_jpl["reference_day"].dt.strftime("%Y-%m-%d")
    )

    df_jpl["sample_day"] = (
        df_jpl["sample_day"].dt.strftime("%Y-%m-%d")
    )

    return df_jpl


# 11. Build the dataset
df_jpl = build_df_jpl(df_A, df_C, df_D, seed=42)


# 12. Validate results
print("Total rows:", len(df_jpl))
print("Total eclipse events:", df_jpl["event_id"].nunique())
print("Positive observations:", (df_jpl["eclipse"] == 1).sum())
print("Negative observations:", (df_jpl["eclipse"] == 0).sum())
print("Available vectors:", df_jpl["vectors_available"].sum())
print("Pending JPL requests:", df_jpl["needs_jpl"].sum())


# 13. Save the dataset
df_jpl.to_csv("df_jpl_manifest.csv", index=False)

df_jpl.head()

## Completing the state vectors

Each observation needs twelve components, six per body. A row already present in D can be reused. For the others we send the requested UT instants to Horizons as Julian dates. The check $t=(\mathrm{JD}-2440587.5)\,86400$ converts the returned day to seconds since the Unix epoch, so a reply of the right length but with different times is rejected.

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from io import StringIO

URL = "https://ssd.jpl.nasa.gov/api/horizons.api"

BODIES = {"sun": "10", "moon": "301"}

STATE = ["x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms"]

ALL_COLS = [
    f"{var}_{body}"
    for body in BODIES
    for var in STATE
]


# 1. Request vectors from JPL
def get_vectors(times, body):

    times = pd.DatetimeIndex(times)

    tlist = " ".join(
        f"'{jd:.9f}'" for jd in times.to_julian_date()
    )

    params = {
        "format": "text",
        "COMMAND": f"'{BODIES[body]}'",
        "MAKE_EPHEM": "'YES'",
        "OBJ_DATA": "'NO'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": "'500@399'",
        "REF_PLANE": "'FRAME'",
        "REF_SYSTEM": "'ICRF'",
        "VEC_TABLE": "'2'",
        "VEC_CORR": "'NONE'",
        "VEC_LABELS": "'NO'",
        "OUT_UNITS": "'KM-S'",
        "TIME_TYPE": "'UT'",
        "TLIST_TYPE": "'JD'",
        "CAL_TYPE": "'GREGORIAN'",
        "CSV_FORMAT": "'YES'",
        "TLIST": tlist
    }

    response = requests.get(URL, params=params, timeout=120)
    response.raise_for_status()

    text = response.text

    if "$$SOE" not in text or "$$EOE" not in text:
        raise ValueError("Invalid JPL response")

    block = text.split("$$SOE")[1].split("$$EOE")[0]

    table = pd.read_csv(StringIO(block), header=None)

    if len(table) != len(times):
        raise ValueError("Unexpected number of JPL records")

    jd = table.iloc[:, 0].to_numpy(dtype=float)

    returned_times = pd.to_datetime(
        np.rint((jd - 2440587.5) * 86400).astype("int64"),
        unit="s"
    )

    if not pd.DatetimeIndex(returned_times).equals(times):
        raise ValueError("JPL timestamps do not match")

    vectors = table.iloc[:, 2:8].astype(float).copy()
    vectors.columns = [f"{var}_{body}" for var in STATE]
    vectors.index = times

    return vectors


# 2. Prepare dataframe
df = df_jpl.copy()

df["datetime_ut"] = pd.to_datetime(df["datetime_ut"])
df = df.set_index("datetime_ut").sort_index()

if df.index.has_duplicates:
    raise ValueError("Duplicated timestamps")

needed = df.index[df["needs_jpl"] == True]

print("Dates to complete:", len(needed))

Each batch contains up to 50 timestamps. A body is queried only where at least one of its six components is missing. The intermediate CSV records progress after each batch. The complete CSV is written once every requested row has a solar and a lunar state.

In [ ]:
# 3. Request missing data in batches
batch_size = 50

for i in range(0, len(needed), batch_size):

    batch = needed[i:i + batch_size]

    for body in BODIES:

        cols = [f"{var}_{body}" for var in STATE]

        missing = batch[
            df.loc[batch, cols].isna().any(axis=1)
        ]

        if len(missing) == 0:
            continue

        vectors = get_vectors(missing, body)

        df.loc[missing, cols] = vectors.to_numpy()

        time.sleep(1)

    df["needs_jpl"] = df[ALL_COLS].isna().any(axis=1)
    df["vectors_available"] = ~df["needs_jpl"]

    df.to_csv("df_jpl_checkpoint.csv")

    print(f"Processed batch {i // batch_size + 1}")


# 4. Verify and save
if df["needs_jpl"].any():
    raise ValueError("Some vectors are still missing")

df_jpl = df.reset_index()
df_jpl.head()
df_jpl.to_csv("df_jpl_complete.csv", index=False)

print("DataFrame completed")

In [ ]:
df_jpl=pd.read_csv("df_jpl_complete.csv")
df_jpl.head()

The resulting windows deliberately concentrate observations near eclipses. A mean position or minimum separation computed over one of these windows need not equal the same quantity measured over an entire day. That difference follows the data into Model 1.